In [95]:
# 라이브러리 설치
# !pip install konlpy

In [96]:
from konlpy.tag import Okt
okt = Okt()
print(okt.morphs('나는 학교에 간다'))

['나', '는', '학교', '에', '간다']


# 데이터의 분할
- KFold의 분할 방식
    - KFold
        - 무작위로 데이터를 폴드화
    - StratifiedKFold
        - 계층화를 유지하면서 폴드화

In [97]:
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold, StratifiedGroupKFold

data = {
    'document' : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'],
    'label' : [1, 1, 0, 0, 1, 0, 0, 1],
    'id' : ['a', 'a', 'b', 'b', 'c', 'c', 'd', 'd']
}
df = pd.DataFrame(data)
df

,document,label,id
0,A,1,a
1,B,1,a
2,C,0,b
3,D,0,b
4,E,1,c
5,F,0,c
6,G,0,d
7,H,1,d


In [98]:
# 일반적인 KFold
X = df['document']
Y = df['label']
groups = df['id']

k_folds = KFold(n_splits=2, shuffle=True, random_state=42)
s_folds = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
sg_folds = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)

In [99]:
for x_idx, y_idx in k_fold.split(X, Y):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
2        C      0  b
3        D      0  b
4        E      1  c
6        G      0  d
  document  label id
0        A      1  a
1        B      1  a
5        F      0  c
7        H      1  d


In [100]:
# 계층화 KFold
for x_idx, y_idx in s_fold.split(X, Y):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    print(df.loc[y_idx])
    break

  document  label id
1        B      1  a
3        D      0  b
6        G      0  d
7        H      1  d
  document  label id
0        A      1  a
2        C      0  b
4        E      1  c
5        F      0  c
  document  label id
0        A      1  a
2        C      0  b
4        E      1  c
5        F      0  c


In [101]:
# 계층별 그룹화 KFold
for x_idx, y_idx in sg_fold.split(X, Y, groups):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
4        E      1  c
5        F      0  c
6        G      0  d
7        H      1  d
  document  label id
0        A      1  a
1        B      1  a
2        C      0  b
3        D      0  b


In [102]:
# 네이버 영화 리뷰 (rating_train.txt)
# pandas 
df = pd.read_csv('../data_git/data_NLP/ratings_train.txt', sep='\t')

In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [104]:
# info()를 통해서 
# 150000개에서 5개의 데이터는 제거 가능
# case1
df.loc[df.isna().any(axis=1)]

,id,document,label
25857,2172111,NaN,1
55737,6369843,NaN,1
110014,1034280,NaN,0
126782,5942978,NaN,0
140721,1034283,NaN,0


In [105]:
df.dropna(inplace=True)

In [106]:
# document가 같은 문장이라면 문장을 하나만 두고 나머지는 제거
df['document'].value_counts()

document
굿                                                                                                  181
good                                                                                                92
최고                                                                                                  85
쓰레기                                                                                                 79
별로                                                                                                  66
                                                                                                  ... 
이런거 만드는 색휘는 진짜 일부로 욕처묵고싶어서 환장한색휘인듯                                                                   1
이영화 평점이 궁금해서 왔는데.너무높다..^^                                                                            1
시청자들에게 사과하세여 제작진..낚시성 예고 지친다..보기싫음 말아라 하고 만듬?조세호씨 때문에 넘웃기게 보는데요 낚시그만하시고 예고와 본방이 이어지는 연출부탁드립니다..      1
여태본 영화중 단연 최고                                                   

In [107]:
# 중복된 데이터는 제서
# 제거하기 전의 데이터의 개수
before = len(df)
# drop_duplicates() : 데이터의 중복을 제거하는 함수
df = df.drop_duplicates(['document']).reset_index(drop=True)
after = len(df)
print("중복 데이터를 제거한 행의 개수 : ", before - after)

중복 데이터를 제거한 행의 개수 :  3813


In [108]:
# id 컬럼의 유일한 데이터들의 길이를 확인
len(df['id'].unique())

146182

In [109]:
len(df)

146182

In [110]:
# label 컬럼의 데이터의 개수를 확인
df['label'].value_counts()

label
0    73342
1    72840
Name: count, dtype: int64

In [111]:
# train, validation, test 데이터셋으로 8:1:1 정도의 비율로 데이터를 분할
# label의 비율에 맞게 데이터를 나눠준다
from sklearn.model_selection import train_test_split
# sklearn에는 3개의 데이터셋으로 나눠주는 함수 존재 X
# train_test_split을 2번 사용
X = df['document'].values
Y = df['label'].values
print(len(X), len(Y))
# test 데이터셋을 10%로 먼저 나눠준다.
X_temp, X_test, Y_temp, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=Y)
# validation 데이터셋을 11% 정도로 나눠준다. (label데이터의 비율에 맞게)
X_train, X_val, Y_train, Y_val  = train_test_split(X_temp, Y_temp, test_size=0.11, random_state=42, stratify=Y_temp)

146182 146182


In [112]:
print(len(X_train)/len(X) * 100)
print(len(X_val)/len(X) * 100)
print(len(X_test) / len(X) * 100)

80.09946505041661
9.89998768658248
10.000547263000916


In [113]:
print(pd.Series(Y_train).value_counts())

0    58746
1    58345
Name: count, dtype: int64


In [114]:
print(pd.Series(Y_val).value_counts())

0    7261
1    7211
Name: count, dtype: int64


In [118]:
# 계층 폴드화 -> 학습 데이터를 분할/학습하여 일반적인 성능을 나타내는 방법
# 폴드화, 하이퍼파라미터 탐색과 같이 사용
folds =[]
# enumerate() -> 리스트에서 위치와 값으로 데이터를 나눠서 되돌려준다.
# s_folds.split(X_train, Y_train) -> 결과값이((tr_idxs, va_idxs))
for fold, (tr_idx, va_idx) in enumerate(s_folds.split(X_train, Y_train)):
    folds.append(
        {
            'fold' : fold,
            'tr_idx' : tr_idx,
            'va_idx' : va_idx
        }
    )

In [119]:
folds

[{'fold': 0,
  'tr_idx': array([     0,      2,      3, ..., 117083, 117086, 117090],
        shape=(58545,)),
  'va_idx': array([     1,      7,      8, ..., 117087, 117088, 117089],
        shape=(58546,))},
 {'fold': 1,
  'tr_idx': array([     1,      7,      8, ..., 117087, 117088, 117089],
        shape=(58546,)),
  'va_idx': array([     0,      2,      3, ..., 117083, 117086, 117090],
        shape=(58545,))}]

In [125]:
# fold에서 첫번째 데이터에서 tr_idx를 가지고 Y_train의 0, 1의 비율을 확인
test_idx = folds[0]['tr_idx']
pd.Series(Y_train[test_idx]).value_counts()

0    29373
1    29172
Name: count, dtype: int64

# 단어의 토큰화
- 문장을 단어로 잘라준다. 
    - 공백을 기준으로 문자를 자른다.
        - 영문에서는 사용 가능, 한글에서는 의미가 소실되는 경우가 발생
    - 형태소를 사용하여 문자를 나눠준다.
        - 국어사전을 로드하여 단어별로 나눠준다.

In [126]:
# 공백을 기반으로 데이터를 나누다.
text = '나는 학교에 간다'

In [127]:
tokens = text.split()
print(tokens)

['나는', '학교에', '간다']


In [128]:
text2 = "Hello World"
tokens2 = text2.split()
print(tokens2)

['Hello', 'World']


In [130]:
from konlpy.tag import Okt
okt = Okt()
print(okt.morphs(text))     # 단어별로 나눠주는 함수

print(okt.pos(text))        # 단어와 단어의 종류
text_pos = okt.pos(text)

['나', '는', '학교', '에', '간다']
[('나', 'Noun'), ('는', 'Josa'), ('학교', 'Noun'), ('에', 'Josa'), ('간다', 'Noun')]


In [139]:
list = []
for t in text_pos:
    if t[1] != 'Josa':
        list.append(t[0])

In [140]:
list

['나', '학교', '간다']

In [145]:
# Okt 로드한 데이터를 이용하여 Okt 형태소 분석
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    _pos = okt.morphs(t)
    print(_pos)
    # if _pos[1] != 'Josa'

['이런', '감동', '...', '삶', '의', '희망이', '된다']
['초등학교', '때', '이', '거', '200', '번', '받음', '..', '정말', '짱']
['아이엠', '옴티머', '스프', '라임']
['나', '만', '재밋', '게', '봤나']
['최고', '의', '영화', '평점', '1', '점주', '는', '초딩', '들', '은', '대체', '뭐', '냐', '?', '요새', '한국', '영화', '들', '보다', '훨', '낫다', '.', '영화', '보고', '10년', '넘게', '기억', '에', '남았던', '명작', '이다', '.']


In [ ]:
# ! pip install Korpora

In [ ]:
# from Korpora import Korpora
# data = Korpora.load('nsmc')

In [ ]:
# ! pip install sentencepiece

In [149]:
# sentencepiece 모듈을 이용하여 형태소 분석
# 모델 학습
# train txt, test txt 파일을 모두 로드하여 학습에 대입
df_tr = pd.read_csv("../data_git/data_NLP/ratings_test.txt", sep ='\t').dropna()
df_te = pd.read_csv("../data_git/data_NLP/ratings_train.txt", sep ='\t').dropna()

In [150]:
# 두개의 데이터프레임을 단순 행결합(union 결합)
total_df = pd.concat([df_tr['document'], df_te['document']], axis = 0, ignore_index=True)

In [152]:
total_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 199992 entries, 0 to 199991
Series name: document
Non-Null Count   Dtype 
--------------   ----- 
199992 non-null  object
dtypes: object(1)
memory usage: 1.5+ MB


In [153]:
# 모델에 학습 시키기 전에 파일로 미리 저장
total_df.to_csv('test.txt', index=False, header=False)

In [156]:
# 모델을 생성
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input = 'test.txt',   # 학습에서 사용할 텍스트 파일
    model_prefix = 'ko_unigram',    # unigram -> 한글에 적합한 형태 (한 단어씩 잘라서 표현) -> 모델명
    vocab_size = 8000,   # 단어사전의 크기(모델의 크기) -> 8000, 16000
    model_type = 'unigram',     # 토큰의 생성 방식 
                                # unigram -> BERT, KoGPT 등에서 사용이 되는 언어 모델 방식
                                # bpe -> GPT2 사용하는 방식(한글에는 부적합)
                                # char -> 문자단위(단어가 너무 짧게 쪼개짐)
                                # word -> 단어 단위(한국어에 부적합)
    character_coverage = 0.9995,     # 학습에 포함할 문자의 종류의 비율
                                    # 1.0인 경우 모든 문자의 종류를 사용
                                    # 0.9995 -> 한글, 영문, 숫자 포함
    input_sentence_size = 1000000,   # 학습 문장을 샘플링
                                    # 모든 데이터를 사용하는게 가장 좋은 방법(시간이 오래 걸림)
                                    # 일부만 샘플링하여 사용하는 방법   
    shuffle_input_sentence = True                  # 샘플링시 문장의 순서를 섞어서 사용
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: test.txt
  input_format: 
  model_prefix: ko_unigram
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 1000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  d

In [158]:
# 생성된 모델을 이용하여 형태소 분석
sp = spm.SentencePieceProcessor()
# 생성된 모델을 로드
sp.load('ko_unigram.model')
text = '나는 학교에 간다'
print(sp.encode(text, out_type=str))

['▁나는', '▁학교', '에', '▁간다']


In [159]:
# ▁ 특수기호는 키보드 입력이 불가
char = '\u2581'
print(char)

▁


In [160]:
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    print(sp.encode(t, out_type=str))

['▁이런', '▁감동', '...', '삶', '의', '▁희망', '이', '▁된다']
['▁초', '등', '학교', '▁때', '▁이거', '▁', '200', '번', '▁받', '음', '..', '▁정말', '▁짱']
['▁아이', '엠', '옴', '티', '머', '스', '프', '라', '임']
['▁나만', '▁재밋게', '▁봤', '나']
['▁최고의', '▁영화', '▁평점', '1', '점주는', '▁초딩', '들은', '▁대체', '▁뭐냐', '?', '▁요새', '▁한국영화', '들', '보다', '▁훨', '▁낫다', '.', '▁영화보고', '▁10', '년', '넘', '게', '▁기억에', '▁남았', '던', '▁명작이다', '.']


In [161]:
from konlpy.tag import Komoran

In [162]:
komoran = Komoran()

In [163]:
print(komoran.morphs(text))     # 형태소 나열
print(komoran.pos(text))        # (형태소, 품사) 튜플 나열
print(komoran.nouns(text))      # 명사만 출력

['나', '는', '학교', '에', '간다']
[('나', 'NP'), ('는', 'JX'), ('학교', 'NNG'), ('에', 'JKB'), ('간다', 'NNP')]
['학교', '간다']


### komoran 동사를 일반적으로 사용하는 것들
- 감성/의도 분석/리뷰 (가장 일반적)
    - NNG(일반명사), NNP(고유명사), VV(동사), VA(형용사), MAG(일반부사), SL(외국어)
- 명사 기반의 분류 (문서에 대한 분류 작업)
    - NNG(일반명사), NNP(고유명사), NR(수사), NP(대명사)
- 의미가 있는 단어를 최대한 포함하고 싶은 경우
    -NNG(일반명사), NNP(고유명사), VV(동사), VA(형용사), MAG(일반부사), MAJ(접속부사), IC(감탄사), SL(외국어)

In [166]:
# 사용할 형태소의 종류
allow_pos = ['NNG', 'NNP', 'VV', 'VA']

# 선택한 형태소를 추출하기 위한 함수를 정의
def komoran_tokenize(text):
    # 선택한 형태소만 저장하는 빈 리스트를 생성
    result = []
    for morph, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(morph)
    return result

for idx, t in enumerate(X_train):
    if idx == 5:
        break
    print(komoran_tokenize(t))

['감동', '삶', '희망', '되']
['초등학교', '때', '받']
[]
['보']
['최고', '영화', '평점', '주', '초딩', '대체', '요새', '한국', '영화', '낫', '영화', '넘', '기억', '남', '명작']


# 벡터화
- 토큰화 작업에서 단어들을 추출했다면 해당 단어들을 숫자형으로 변환
    - 숫자형태로 변환하는 이유는? -> 컴퓨터가 숫자로만 연산이 가능하기 때문에
- 숫자형태로 변환한 데이터를 학습 데이터로 이용, 정답은 label 데이터를 이용하여 규칙을 생성해가는 과정

In [170]:
# one-hot encoding -> 하나의 리뷰에서 특정 단어가 포함되어 있는가?
df = pd.read_csv('../data_git/data_NLP/ratings_train.txt', sep='\t').dropna()
df.drop('id', axis=1, inplace=True)
df.head()


,document,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,너무재밓었다그래서보는것을추천한다,0
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [171]:
# 전체의 텍스트를 이용해서 학습을 통한 단어를 습득한 뒤
# 해당하는 단어들이 리뷰에 포함되어있는가?

from sklearn.feature_extraction.text import CountVectorizer

In [179]:
vectorizer = CountVectorizer(binary=True)
# 학습을 한 뒤 변환(data 대입) -> 데이터는 document에서 10개의 데이터
X = vectorizer.fit_transform(df['document'].head(5))
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 29 stored elements and shape (5, 29)>

In [180]:
# 학습한 단어들이 무엇인가 출력 (단어 사전)
vocab = vectorizer.get_feature_names_out()
print(vocab)

['가볍지' '교도소' '너무나도' '너무재밓었다그래서보는것을추천한다' '늙어보이기만' '더빙' '던스트가' '돋보였던' '목소리'
 '사이몬페그의' '솔직히' '스파이더맨에서' '않구나' '없다' '연기가' '영화' '오버연기조차' '이뻐보였다' '이야기구먼'
 '익살스런' '재미는' '조정' '진짜' '짜증나네요' '초딩영화줄' '커스틴' '평점' '포스터보고' '했던']


In [181]:
# get_feature_names_out()의 단어를 포함하고 있는지 확인
print(X.toarray())

[[0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 1 1 0 0 0 0 1 0 0]
 [0 0 1 0 1 0 1 1 0 1 0 1 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 0 1]]


In [194]:
# Okt + CounterVectorizer 같이 사용 -> 토큰화 + 벡터화

okt = Okt()

# 형태소 변환 함수 정의
def okt_tokenize(text):
    # 특정 형태의 단어들만 추출한다.
    # 명사, 동사, 형용사만 선택
    select_pos = ['Noun', 'Verb', 'Adjective']
    # (단어, 형태)를 출력하는 pos()함수 이용
    # result = okt.morphs(text)
    result = [
        word for word, pos in okt.pos(text) if pos in select_pos
    ]
    # 위의 코드의 작동 방식
    # result2 = []
    # for word, pos in okt.pos(text):
    #     if pos in select_pos:
    #         result2.append(word)
    return result

# CounterVertorizer 생성
vectorizer_okt = CountVectorizer(
    tokenizer=okt_tokenize,
    lowercase = False,
    binary=True
)
x_okt = vectorizer_okt.fit_transform(df['document'].head(5))

In [195]:
print(vectorizer_okt.get_feature_names_out())

['가볍지' '교도소' '구먼' '늙어' '다그' '더빙' '던스트' '돋보였던' '래서' '목소리' '몬페' '무재' '밓었'
 '보고' '보는것을' '보였다' '보이기만' '솔직히' '스파이더맨' '않구나' '없다' '연기' '영화' '오버' '의' '이뻐'
 '이야기' '익살스런' '재미' '조정' '줄' '진짜' '짜증나네요' '초딩' '추천' '커스틴' '평점' '포스터' '했던'
 '흠']


In [196]:
print(x_okt.toarray())

[[0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0
  0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 0 1 0 0
  0 1 0 1]
 [0 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0
  1 0 0 0]
 [0 0 0 1 0 0 1 1 0 0 1 0 0 0 0 1 1 0 1 0 0 1 1 0 1 1 0 1 0 0 0 0 0 0 0 1
  0 0 1 0]]


In [197]:
pd.DataFrame(
    x_okt.toarray(),
    columns = vectorizer_okt.get_feature_names_out()
)

,가볍지,교도소,구먼,늙어,다그,더빙,던스트,돋보였던,래서,목소리,몬페,무재,밓었,보고,보는것을,보였다,보이기만,솔직히,스파이더맨,않구나,없다,연기,영화,오버,의,이뻐,이야기,익살스런,재미,조정,줄,진짜,짜증나네요,초딩,추천,커스틴,평점,포스터,했던,흠
0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1
2,0,0,0,0,1,0,0,0,1,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0
4,0,0,0,1,0,0,1,1,0,0,1,0,0,0,0,1,1,0,1,0,0,1,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0
